---
# Relatório de Análise e Consolidação: Preço dos Combustíveis 
(Silver -> Gold)

---

### Visão Geral do Notebook
Este notebook é responsável pelo processamento, agregação e consolidação dos dados de preços de combustíveis pré-tratados na camada **Silver**. O objetivo principal é gerar métricas analíticas e estatísticas avançadas (médias, variações de preço, desvio padrão, min/máx) para apoiar dashboards de inteligência de negócios e relatórios estratégicos sobre a volatilidade do mercado de combustíveis no Brasil.

---

### Arquitetura e Fluxo de Dados
* **Origem :** `workspace.mvp-engenharia-dados.base_combustivel_silver`
* **Processamento:** PySpark SQL / Window Functions
* **Destino (Outbound / Gold):** Tabelas agregadas para consumo via BI (Power BI / Databricks SQL)

---

### Bibliotecas & Funções Utilizadas
* **`pyspark.sql.functions`**:
  * `col`: Manipulação de colunas no DataFrame.
  * `avg`, `spark_round`: Cálculo e formatação de médias financeiras.
  * `min`, `max`, `stddev`: Análise de dispersão e valores extremos por região e produto.
  * `lag`: Cálculo de variações temporais (variação MoM/WoW de preço de venda).
  * `when`, `count`: Agregadores e regras condicionais para verificação de cobertura.
* **`pyspark.sql.window`**:
  * `Window`: Particionamento temporário e geográfico para cálculo de métricas acumuladas e janelas deslizantes.

---

### Estrutura da Base Consumida (Silver)
A tabela Silver consumida traz registros detalhados das coletas realizadas pela ANP contendo os seguintes atributos principais:
* **Dimensão Geográfica:** `regiao_sigla`, `estado_sigla`, `municipio`, `bairro`
* **Dimensão Estabelecimento:** `revenda`, `cnpj_revenda`, `bandeira`
* **Dimensão Produto & Tempo:** `produto`, `data_coleta`, `ano_mes`, `ano`, `mes`
* **Métricas Financeiras:** `valor_venda`, `valor_compra`, `unidade_medida`

---

### Etapas do Processamento
1. **Ingestão:** Leitura dos dados higienizados da tabela `base_combustivel_silver`.
2. **Transformação & Métricas:**
   * Agregação de preço médio de venda e compra por Região, Estado e Produto.
   * Cálculo de variação do preço ao longo dos meses utilizando janelas móveis (`lag`).
   * Apuração de desvio padrão para identificação de discrepâncias regionais de preço.
3. **Carga (Gold):** Disponibilização dos dados agregados para consumo final.
---

In [0]:
# Import
from pyspark.sql.functions import col, avg, round as spark_round, lag, min as spark_min, max as spark_max, stddev, count, when
from pyspark.sql.window import Window

# Ingestão da base Silver
df_preco_combustivel = spark.read.table("workspace.`mvp-engenharia-dados`.base_combustivel_silver")

In [0]:
# Leitura da base
display(df_preco_combustivel.limit(10))

regiao_sigla,estado_sigla,municipio,revenda,cnpj_revenda,bairro,produto,data_coleta,valor_venda,valor_compra,unidade_medida,bandeira,ano_mes,ano,mes
NE,PE,RECIFE,POSTO AÃORES LTDA,05.930.453/0001-30,HIPODROMO,GASOLINA,2023-01-03,4.890,null,R$ / litro,TDC DISTRIBUIDORA,2023-01,2023,1
SE,SP,ARACATUBA,SANCHES WEILER COMERCIO DE COMBUSTIVEIS LTDA,07.726.889/0001-38,JARDIM PAULISTA,GASOLINA,2023-01-03,4.950,null,R$ / litro,VIBRA ENERGIA,2023-01,2023,1
SE,SP,BARUERI,CENTRO AUTOMOTIVO CRS LTDA,03.877.914/0001-23,VILA SAO SILVESTRE,ETANOL,2023-01-03,4.190,null,R$ / litro,RAIZEN,2023-01,2023,1
SE,SP,CAMPINAS,AUTO POSTO 2002 DE CAMPINAS LTDA,01.257.449/0001-00,JD PARAISO,GASOLINA,2023-01-02,4.990,null,R$ / litro,IPIRANGA,2023-01,2023,1
SE,SP,AMPARO,ARGEU GUARIZZO & CIA LTDA,02.051.228/0001-36,CENTRO,ETANOL,2023-01-05,3.990,null,R$ / litro,RAIZEN,2023-01,2023,1
CO,MS,CAMPO GRANDE,AUTO POSTO ECOLOGICO LTDA,18.947.284/0001-93,JARDIM VERA CRUZ,ETANOL,2023-01-05,3.990,null,R$ / litro,IPIRANGA,2023-01,2023,1
NE,CE,CRATO,FERNANDO LEITE DE LACERDA,04.682.155/0001-06,SAO MIGUEL,GASOLINA,2023-01-11,5.490,null,R$ / litro,VIBRA ENERGIA,2023-01,2023,1
NE,CE,SOBRAL,V.C.EMPREENDIMENTOS LTDA,03.551.935/0002-35,CORACAO DE JESUS,ETANOL,2023-01-09,4.690,null,R$ / litro,RAIZEN,2023-01,2023,1
CO,GO,RIO VERDE,POSTO BARRINHAS LTDA,01.301.913/0001-00,PARQUE BANDEIRANTES,ETANOL,2023-01-10,3.940,null,R$ / litro,VIBRA ENERGIA,2023-01,2023,1
CO,MT,CACERES,COMLUC - COMERCIO DE LUBRIFICANTES E COMBUSTIVEIS LTDA,03.927.792/0001-32,JD SAO LUIZ,GASOLINA,2023-01-09,5.090,null,R$ / litro,RAIZEN,2023-01,2023,1


In [0]:
# Variação Média do COmbustível por Estado (UF) e Mês/Ano

# 1. Filtra Gasolina e Etanol
df_gasolina_etanol = df_preco_combustivel.filter(col("produto").isin("GASOLINA", "ETANOL"))

# 2. Calcula a média mensal por Estado e Produto
df_media_mensal = df_gasolina_etanol \
    .groupBy("estado_sigla", "produto", "ano_mes") \
    .agg(spark_round(avg("valor_venda"), 3).alias("preco_medio"))

# 3. Define a janela ordenada pelo tempo para cada Estado e Produto
janela_estado_produto = Window.partitionBy("estado_sigla", "produto").orderBy("ano_mes")

# 4. Calcula o preço do mês anterior e a variação percentual (%)
df_gold_variacao_combustiveis = df_media_mensal \
    .withColumn("preco_mes_anterior", lag("preco_medio", 1).over(janela_estado_produto)) \
    .withColumn("variacao_percentual", 
                spark_round(((col("preco_medio") - col("preco_mes_anterior")) / col("preco_mes_anterior")) * 100, 2)) \
    .orderBy("estado_sigla", "produto", "ano_mes")

print("---\nVARIAÇÃO MÉDIA DA GASOLINA E DO ETANOL POR ESTADO E MÊS/ANO")
display(df_gold_variacao_combustiveis.limit(10))

---
VARIAÇÃO MÉDIA DA GASOLINA E DO ETANOL POR ESTADO E MÊS/ANO


estado_sigla,produto,ano_mes,preco_medio,preco_mes_anterior,variacao_percentual
AC,ETANOL,2023-01,4.367,null,null
AC,ETANOL,2023-02,4.438,4.367,1.63
AC,ETANOL,2023-03,4.439,4.438,0.02
AC,ETANOL,2023-04,4.426,4.439,-0.29
AC,ETANOL,2023-05,4.790,4.426,8.22
AC,ETANOL,2023-06,4.777,4.790,-0.27
AC,ETANOL,2023-07,4.832,4.777,1.15
AC,ETANOL,2023-08,4.850,4.832,0.37
AC,ETANOL,2023-09,4.858,4.850,0.16
AC,ETANOL,2023-10,4.811,4.858,-0.97


In [0]:
# Paridade Percentual entre Etanol e Gasolina por Estado
# Aqui, representa a relação percentual de preço entre o Etanol e a Gasolina em cada Estado.

# 1. Filtra Etanol e Gasolina
df_etanol_gasolina = df_preco_combustivel.filter(col("produto").isin("ETANOL", "GASOLINA"))

# 2. Calcula o preço médio de cada produto por Estado
df_precos_estado = df_etanol_gasolina \
    .groupBy("estado_sigla", "produto") \
    .agg(avg("valor_venda").alias("preco_medio"))

# 3. Pivot para transformar os produtos em colunas
df_pivot = df_precos_estado \
    .groupBy("estado_sigla") \
    .pivot("produto", ["ETANOL", "GASOLINA"]) \
    .avg("preco_medio")

# 4. Calcula a paridade e a indicação de consumo
df_gold_paridade = df_pivot \
    .withColumn("preco_medio_etanol", spark_round(col("ETANOL"), 3)) \
    .withColumn("preco_medio_gasolina", spark_round(col("GASOLINA"), 3)) \
    .withColumn("paridade_percentual", spark_round((col("ETANOL") / col("GASOLINA")) * 100, 2)) \
    .withColumn("recomendacao", 
                when(col("paridade_percentual") <= 70.0, "Compensa ETANOL")
                .otherwise("Compensa GASOLINA")) \
    .select("estado_sigla", "preco_medio_etanol", "preco_medio_gasolina", "paridade_percentual", "recomendacao") \
    .orderBy("paridade_percentual")

print("---\nPARIDADE ETANOL x GASOLINA POR ESTADO")
display(df_gold_paridade.limit(10))

---
PARIDADE ETANOL x GASOLINA POR ESTADO


estado_sigla,preco_medio_etanol,preco_medio_gasolina,paridade_percentual,recomendacao
MT,3.855,6.038,63.84,Compensa ETANOL
MS,3.949,5.954,66.33,Compensa ETANOL
SP,3.874,5.827,66.49,Compensa ETANOL
DF,4.124,6.004,68.69,Compensa ETANOL
GO,4.111,5.979,68.77,Compensa ETANOL
PR,4.215,6.092,69.19,Compensa ETANOL
MG,4.063,5.852,69.43,Compensa ETANOL
AM,4.916,6.828,72.00,Compensa GASOLINA
AC,5.110,7.088,72.10,Compensa GASOLINA
ES,4.451,6.039,73.71,Compensa GASOLINA


In [0]:
# Maior Variação de Preço por Bandeira e Região

df_bandeiras = df_preco_combustivel.filter(col("bandeira").isNotNull())

df_gold_variacao_bandeira = df_bandeiras \
    .groupBy("regiao_sigla", "bandeira") \
    .agg(
        count("*").alias("total_pesquisas"),
        spark_round(spark_min("valor_venda"), 2).alias("preco_minimo"),
        spark_round(spark_max("valor_venda"), 2).alias("preco_maximo"),
        spark_round(spark_max("valor_venda") - spark_min("valor_venda"), 2).alias("amplitude_variacao_rs"),
        spark_round(stddev("valor_venda"), 3).alias("desvio_padrao_preco")
    ) \
    .orderBy("regiao_sigla", col("amplitude_variacao_rs").desc())

print("---\nVARIAÇÃO DE PREÇO POR BANDEIRA E REGIÃO")
display(df_gold_variacao_bandeira.limit(10))

---
VARIAÇÃO DE PREÇO POR BANDEIRA E REGIÃO


regiao_sigla,bandeira,total_pesquisas,preco_minimo,preco_maximo,amplitude_variacao_rs,desvio_padrao_preco
CO,VIBRA,21641,2.75,7.55,4.80,1.106
CO,BRANCA,46715,2.69,7.49,4.80,1.074
CO,RAIZEN,13133,2.69,7.39,4.70,1.1
CO,IPIRANGA,24019,2.67,7.29,4.62,1.087
CO,IDAZA,246,2.99,7.29,4.30,1.182
CO,ALE,2015,3.09,7.29,4.20,1.037
CO,TAURUS,2949,2.96,7.05,4.09,1.059
CO,CIAPETRO,355,2.99,6.94,3.95,1.057
CO,VIBRA ENERGIA,3274,3.07,6.69,3.62,0.829
CO,MASUT DISTRIBUIDORA,588,3.19,6.79,3.60,0.99


In [0]:
# Definindo o nome completo da tabela no Unity Catalog (Catalog.Schema.Tabela)
nome_tabela_gold_1 = "workspace.`mvp-engenharia-dados`.combustivel_variacao_gold"

# Salvando o DataFrame como Tabela Delta no Unity Catalog
df_gold_variacao_combustiveis.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(nome_tabela_gold_1)

print("Tabela 'combustivel_variacao_gold' salva com sucesso no Unity Catalog!")

Tabela 'combustivel_variacao_gold' salva com sucesso no Unity Catalog!


In [0]:
# Definindo o nome completo da tabela no Unity Catalog (Catalog.Schema.Tabela)
nome_tabela_gold_2 = "workspace.`mvp-engenharia-dados`.combustivel_paridade_gold"

# Salvando o DataFrame como Tabela Delta no Unity Catalog
df_gold_paridade.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(nome_tabela_gold_2)

print("Tabela 'combustivel_paridade_gold' salva com sucesso no Unity Catalog!")

Tabela 'combustivel_paridade_gold' salva com sucesso no Unity Catalog!


In [0]:
# Definindo o nome completo da tabela no Unity Catalog (Catalog.Schema.Tabela)
nome_tabela_gold_3 = "workspace.`mvp-engenharia-dados`.combustivel_bandeira_gold"

# Salvando o DataFrame como Tabela Delta no Unity Catalog
df_gold_variacao_bandeira.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(nome_tabela_gold_3)

print("Tabela 'combustivel_bandeira_gold' salva com sucesso no Unity Catalog!")

Tabela 'combustivel_bandeira_gold' salva com sucesso no Unity Catalog!
